# Module 15 — Python Programming for Data Handling

**Course:** Programming Fundamentals Using Python

**Prerequisites:** Modules 7–12 (lists, dictionaries, functions, files,
exceptions). Module 14 (classes) is *not* needed here — everything in this
module uses plain lists, dictionaries and functions, exactly the data
shapes you will meet again in Pandas and NumPy later in your degree.

**What you will learn:**

- Structured vs unstructured data
- Representing records as dictionaries in lists
- Processing, filtering, searching and summarising collections of records
- Reading CSV data with the standard-library `csv` module
- Cleaning messy data using core Python only
- Reading and writing JSON with the standard-library `json` module
- A complete **Student Performance Data Analyzer** project

| Previous | Next |
|---|---|
| [Module 14 — Introduction to OOP](14_Introduction_to_OOP.ipynb) | [Module 16 — Computational Thinking](16_Computational_Thinking.ipynb) |

---

**Learning objectives** — after this module you can:

1. Choose a sensible data structure for tabular data and defend the choice.
2. Load, clean, filter and summarise records using only core Python.
3. Read and write CSV files with `csv.reader`, `csv.writer`,
   `csv.DictReader` and `csv.DictWriter`.
4. Read and write JSON files with `json.load`, `json.dump`,
   `json.loads` and `json.dumps`.
5. Build a small end-to-end data program: load → validate → clean →
   analyse → report → save.

**How to use this notebook:** run every cell in order (Kernel ▸ Restart
Kernel and Run All Cells). Cells marked *Practice* are yours to modify —
break them and fix them. Exercises at the end have solutions hidden in
collapsible details blocks; try seriously hard before opening them.

## Contents

1. [Structured vs unstructured data](#1)
2. [Records: lists of dictionaries](#2)
3. [Processing collections of records](#3)
4. [Reading CSV files](#4)
5. [Basic data cleaning](#5)
6. [Aggregation and summary statistics](#6)
7. [JSON](#7)
8. [Project — Student Performance Data Analyzer](#8)
9. [Lab 15](#9)
10. [Common mistakes](#10)
11. [Exercises](#11)
12. [Summary](#12)

<a id="1"></a>

## 1. Structured vs unstructured data

### Concept

**Unstructured data** has no predictable shape: the text of an essay, a
photo, a free-form comment. **Structured data** follows a fixed scheme:
every item has the same kind of pieces in the same places.

A **mark sheet** is structured: every row is a student, and every student
has the same fields — name, roll number, marks. Schemes like this are why
spreadsheets, databases, CSV files and DataFrames all look similar.

### Why do we need it?

Data Science is mostly *structured-data plumbing*: load records, check
them, fix them, compute summaries. Before a single library is involved,
the questions are always the same core-Python questions:

- How do I store one record? → a **dictionary** (field name → value).
- How do I store many records? → a **list** of those dictionaries.
- How do I compute over them? → **loops + accumulators** (Module 6),
  wrapped in **functions** (Module 9).

### Example — two kinds of data

The first string below is unstructured: useful for humans, painful for
programs. The list of dictionaries below it is structured: each record
has the *same keys in the same roles*, so any program can walk it.

In [ ]:
unstructured = "Ali got 78 in Math and 84 in CS; Sara got 91 in Math."

# Structured: every record has the same shape.
students = [
    {"name": "Ali", "math": 78, "cs": 84},
    {"name": "Sara", "math": 91, "cs": 88},
    {"name": "Omar", "math": 65, "cs": 70},
]

print("Unstructured — a program must 'understand' English to use it:")
print("   ", unstructured)
print()
print("Structured — a program can compute directly:")
for student in students:
    print("   ", student["name"], "→ math:", student["math"], "cs:", student["cs"])

**What the code does:**

- `students` is a **list** (ordered collection — Module 7) whose elements
  are **dictionaries** (key → value — Module 8).
- One record = one dictionary. Keys are the *field names*; values are the
  *data*. Every record repeats the same keys — that uniformity is the
  whole point.
- The loop treats every record identically: `student["name"]` works for
  every element because the scheme guarantees the key exists.

<a id="2"></a>

## 2. Records: lists of dictionaries

### Concept

The pattern `list_of_dicts` is *the* beginner representation of a table:

```python
students = [
    {"name": "Ali", "marks": 78},
    {"name": "Sara", "marks": 91},
    {"name": "Ahmed", "marks": 65},
]
```

Read it as: "students is a table; each `{...}` is one row; each key is a
column." Mapping to spreadsheet vocabulary:

| Table idea | Python here |
|---|---|
| Table | the list `students` |
| Row / record | one dictionary, e.g. `students[0]` |
| Column / field | a key, e.g. `"marks"` |
| Cell | `students[0]["marks"]` |

### Why do we need it?

A **parallel lists** layout (`names = [...]`, `marks = [...]`) forces you
to keep indices in sync — add one student in one list and forget the
other, and your data is silently corrupt. A list of dictionaries bundles
each record's fields *with* the record, so there is nothing to keep in
sync. It is also exactly the shape `csv.DictReader` and JSON give you.

### Beginner example — one record, then many

In [ ]:
# One record: a dictionary maps field names to values.
student = {"name": "Ali", "roll_no": "DS-101", "marks": 78}
print("Name:     ", student["name"])
print("Roll no:  ", student["roll_no"])
print("Marks:    ", student["marks"])

# Many records: a list of such dictionaries.
students = [
    {"name": "Ali", "roll_no": "DS-101", "marks": 78},
    {"name": "Sara", "roll_no": "DS-102", "marks": 91},
    {"name": "Ahmed", "roll_no": "DS-103", "marks": 65},
]

# Print a small table (f-string alignment from Module 3).
print()
print(f"{'Name':<8}{'Roll':<10}{'Marks':>5}")
for student in students:
    print(f"{student['name']:<8}{student['roll_no']:<10}{student['marks']:>5}")

**Line by line:**

- `student = {...}` builds one record; `student["name"]` looks up a field
  by its key (Module 8).
- `students = [ {...}, {...}, {...} ]` is a list of three records.
- The `for` loop (Module 6) visits one record at a time; the loop
  variable `student` is *the current dictionary*.
- `f"{student['name']:<8}"` pads the value to width 8, left-aligned —
  note the **single quotes inside the f-string** because the f-string
  itself uses double quotes.

### Practical example — adding, updating, removing records

A list of dictionaries is fully mutable at both levels: the list can
grow/shrink (Module 7) and each record's fields can change (Module 8).

In [ ]:
students = [
    {"name": "Ali", "roll_no": "DS-101", "marks": 78},
    {"name": "Sara", "roll_no": "DS-102", "marks": 91},
]

# Add a record (append to the list).
students.append({"name": "Omar", "roll_no": "DS-103", "marks": 42})

# Update a field (assign into the dictionary).
students[0]["marks"] = 82

# Remove a record (remove the dictionary from the list).
for student in students:
    if student["name"] == "Omar":
        students.remove(student)
        break

print("After append, update and remove:")
for student in students:
    print("  ", student)

**Explanation:** `students.remove(student)` removes the dictionary
currently referenced by the loop variable — a `break` right after is
essential, because removing while still looping is exactly the
modification-during-iteration trap from Module 7.

### Challenge example — records with nested structure

Real records nest: a student has *marks in several subjects*. A
dictionary value can itself be a list (or another dictionary).

In [ ]:
students = [
    {"name": "Ali", "marks": {"math": 78, "cs": 84, "stats": 90}},
    {"name": "Sara", "marks": {"math": 91, "cs": 88, "stats": 95}},
    {"name": "Omar", "marks": {"math": 42, "cs": 55, "stats": 60}},
]

for student in students:
    subjects = student["marks"]              # the nested dictionary
    subject_count = len(subjects)
    total = sum(subjects.values())           # sum of the values
    average = total / subject_count
    print(f"{student['name']:<6} total={total:>3}  average={average:.1f}")

`student["marks"]` reaches into the nested dictionary; `.values()` gives
the mark values (Module 8); `sum()` and `len()` do the arithmetic. Two
levels of lookup — `student["marks"]["math"]` — would fetch one subject.

<a id="3"></a>

## 3. Processing collections of records

### Concept

Almost every data task is one of a handful of patterns over a list of
records:

| Pattern | Shape |
|---|---|
| **Filter** | keep records matching a condition |
| **Search** | find the *first* record matching a condition |
| **Transform** | build a new value for each record |
| **Aggregate** | reduce all records to one number (sum, average, max…) |
| **Count** | tally how many records fall in each category |

Each is a `for` loop with an accumulator (Modules 6–9) — no libraries.

### Why do we need it?

These five patterns *are* what Pandas calls `filter`, `apply`,
`sum`/`mean`/`max` and `value_counts`. Learning them by hand first means
that later, when Pandas does them in one line, you will know exactly
what the library is doing for you.

### Filtering — records matching a condition

In [ ]:
students = [
    {"name": "Ali", "marks": 78},
    {"name": "Sara", "marks": 91},
    {"name": "Ahmed", "marks": 65},
    {"name": "Omar", "marks": 45},
    {"name": "Hina", "marks": 84},
]

passed = []                     # accumulator: list of records
for student in students:
    if student["marks"] >= 50:  # the filter condition
        passed.append(student)

print("Students who passed:")
for student in passed:
    print("  ", student["name"], "-", student["marks"])

**Explanation:** start with an empty accumulator list; the `if` tests
each record's `marks` field; matching records are appended. The original
`students` list is untouched — filtering *derives* a new list.

### Searching — the first matching record

Search differs from filter in two ways: it stops at the first hit, and
it must handle **not finding anything**.

In [ ]:
students = [
    {"name": "Ali", "roll_no": "DS-101", "marks": 78},
    {"name": "Sara", "roll_no": "DS-102", "marks": 91},
    {"name": "Ahmed", "roll_no": "DS-103", "marks": 65},
]

def find_by_name(records, name):
    """Return the first record whose 'name' is name, or None."""
    for record in records:
        if record["name"] == name:
            return record        # found: stop immediately
    return None                  # loop finished: nothing matched


result = find_by_name(students, "Sara")
print("Search 'Sara':  ", result)

result = find_by_name(students, "Zoya")
print("Search 'Zoya':  ", result)

The `return` inside the loop stops the search at the first match — no
wasted work. Returning `None` for "not found" is a Python convention
callers can test with `is None`.

### Transforming — building new values from records

In [ ]:
students = [
    {"name": "Ali", "marks": 78},
    {"name": "Sara", "marks": 91},
    {"name": "Omar", "marks": 45},
]

# Transform each record into a report line (string).
lines = []
for student in students:
    grade = "A" if student["marks"] >= 80 else ("B" if student["marks"] >= 60 else "C")
    lines.append(f"{student['name']}: marks {student['marks']}, grade {grade}")

for line in lines:
    print(line)

The chained conditional expression assigns one of three grades — the
ternary from Module 5, nested once. Writing the same thing with plain
`if/elif/else` statements is equally correct and often clearer:

```python
if student["marks"] >= 80:
    grade = "A"
elif student["marks"] >= 60:
    grade = "B"
else:
    grade = "C"
```

### Challenge example — filtering on a nested field

Combine nested access with filtering: students whose *math* mark is at
least 85.

In [ ]:
students = [
    {"name": "Ali", "marks": {"math": 78, "cs": 84}},
    {"name": "Sara", "marks": {"math": 91, "cs": 88}},
    {"name": "Omar", "marks": {"math": 42, "cs": 55}},
    {"name": "Hina", "marks": {"math": 86, "cs": 80}},
]

math_stars = []
for student in students:
    if student["marks"]["math"] >= 85:       # nested lookup inside the filter
        math_stars.append(student["name"])

print("Math stars:", math_stars)

<a id="4"></a>

## 4. Reading CSV files

### Concept

**CSV — Comma-Separated Values** — is the most common file format for
tabular data: one row per line, fields separated by commas. The first
line often holds the column names (the *header*):

```csv
name,roll_no,marks
Ali,DS-101,78
Sara,DS-102,91
```

Python's standard library ships a `csv` module that parses these files
correctly — including quoted fields like `"Khan, Ali"` that a naive
`line.split(",")` would break.

### Why do we need it?

Exports from spreadsheets, databases, form tools and LMS platforms are
overwhelmingly CSV. Reading them *is* the first step of nearly every
analysis you will ever run.

### Beginner example — writing a CSV, then reading it back

In [ ]:
import csv
from pathlib import Path

DATA_DIR = Path("file_sandbox")          # one folder for this notebook's files
DATA_DIR.mkdir(exist_ok=True)            # create it if missing (Module 11)
csv_path = DATA_DIR / "students.csv"

# --- write a CSV with csv.writer ---
with open(csv_path, "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["name", "roll_no", "marks"])      # header row
    writer.writerow(["Ali", "DS-101", 78])
    writer.writerow(["Sara", "DS-102", 91])
    writer.writerow(["Ahmed", "DS-103", 65])

print("Wrote", csv_path.as_posix())

# --- read it back with csv.reader ---
with open(csv_path, "r", newline="", encoding="utf-8") as file:
    reader = csv.reader(file)
    for row in reader:                    # each row is a list of strings
        print("  ", row)

**Line by line:**

- `open(csv_path, "w", newline="", encoding="utf-8")` — the `with`
  pattern from Module 11; `newline=""` is the csv module's recommended
  setting and prevents blank lines on Windows; `encoding="utf-8"` makes
  the file portable across machines.
- `csv.writer(file)` wraps the file; each `writerow(...)` call writes one
  list as one CSV line, inserting commas (and quotes) where needed.
- `csv.reader(file)` yields one **list of strings** per row — notice
  `"78"` came back as the *string* `"78"`, not the number. CSV has no
  types; conversion back to `int` is your job (see cleaning, §5).

### DictReader — rows as dictionaries

`csv.DictReader` uses the header line as keys and yields each row as a
**dictionary** — records straight into the pattern from §2.

In [ ]:
with open(csv_path, "r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    for record in reader:
        print(record["name"], "studied", record["roll_no"], "and scored", record["marks"])

No index bookkeeping: each `record` is a dictionary keyed by the column
names. `record["marks"]` is still a *string* — again, cleaning converts.

### DictWriter — dictionaries out to CSV

In [ ]:
records = [
    {"name": "Ali", "roll_no": "DS-101", "marks": 78},
    {"name": "Sara", "roll_no": "DS-102", "marks": 91},
]

with open(csv_path, "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=["name", "roll_no", "marks"])
    writer.writeheader()                 # writes the keys as the header row
    writer.writerows(records)            # writes every dictionary in one call

# Prove it round-tripped: read the raw text back.
print(csv_path.read_text(encoding="utf-8"))

`fieldnames` fixes the column order and which keys get written — extra
keys in a record would simply be ignored.

### Practical example — load records from CSV into the list-of-dicts form

In [ ]:
# Start from the file written above; build typed records in memory.
students = []                            # will hold real records
with open(csv_path, "r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    for row in reader:
        students.append({
            "name": row["name"],
            "roll_no": row["roll_no"],
            "marks": int(row["marks"]),  # convert the string to a number
        })

print("Loaded", len(students), "records with typed marks:")
for student in students:
    print("  ", student)

This load-and-convert loop is the single most reused snippet of the
module — it appears in the project below almost unchanged.

### Challenge example — CSV with a numeric filter

Students earning 80 or more, straight from the file:

In [ ]:
with open(csv_path, "r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    high_scorers = []
    for row in reader:
        if int(row["marks"]) >= 80:
            high_scorers.append(row["name"])

print("High scorers:", high_scorers)

<a id="5"></a>

## 5. Basic data cleaning

### Concept

Real data arrives messy. Recurring problems and their core-Python fixes:

| Problem | Fix (module where you learnt it) |
|---|---|
| stray spaces around values | `strip()` (Module 4) |
| empty / missing values | explicit check `if not value:` (Module 5) |
| numbers stored as text | `int()` / `float()` (Module 2) |
| inconsistent capitalisation | `lower()` / `title()` (Module 4) |
| duplicate records | compare with records already accepted (Module 7) |
| invalid values (marks = 999) | range checks, skip or flag (Module 5) |

**Cleaning rule of thumb:** validate *first*, convert *second*, append
*last* — never let a doubtful record into your clean list.

### Beginner example — strip and standardise text

In [ ]:
messy_names = ["  ali ", "SARA", "Ahmed ", "  hina"]
clean_names = []
for name in messy_names:
    clean_names.append(name.strip().title())     # remove spaces, then Title Case

print("Before:", messy_names)
print("After: ", clean_names)

`name.strip()` trims whitespace (spaces, tabs, newlines) from both ends; `.title()` capitalises
each word. Chaining the two (Module 4) is idiomatic and readable.

### Practical example — validating and converting records

The full cleaning pipeline on a deliberately dirty dataset:

In [ ]:
dirty_rows = [
    ["Ali", "78"],          # fine
    ["  sara ", "91"],      # spaces, lowercase
    ["Ahmed", ""],          # missing marks
    ["Hina", "84"],         # fine
    ["Omar", "999"],        # impossible marks
    ["Ali", "78"],          # duplicate of the first row
]

clean_students = []
for row in dirty_rows:
    name = row[0].strip().title()        # 1. standardise text
    marks_text = row[1].strip()

    if name == "" or marks_text == "":   # 2. reject missing values
        print("Skipping (missing value):", row)
        continue
    if not marks_text.isdigit():         # 3. reject non-numeric marks
        print("Skipping (not numeric):", row)
        continue

    marks = int(marks_text)              # 4. convert, now that it is safe
    if marks < 0 or marks > 100:         # 5. reject out-of-range values
        print("Skipping (out of range):", row)
        continue

    # 6. reject duplicates by comparing with what we already accepted
    duplicate = False
    for existing in clean_students:
        if existing["name"] == name and existing["marks"] == marks:
            duplicate = True
            break
    if duplicate:
        print("Skipping (duplicate):", row)
        continue

    clean_students.append({"name": name, "marks": marks})

print()
print("Clean records:")
for student in clean_students:
    print("  ", student)

**Walk through the logic:**

- `strip().title()` normalises names, so `"  sara "` and `"SARA"` both
  become `"Sara"`.
- `isdigit()` (Module 4) guards the `int()` conversion — calling
  `int("")` would crash with a `ValueError`.
- Range check before accepting: 999 is numeric but impossible.
- The duplicate check loops over the *clean* list, comparing both fields.
- Every rejected row is **reported**, never silently dropped — silent
  data loss is the cardinal sin of data cleaning.

### Challenge example — cleaning CSV rows

The same pipeline reading from a file, as it would in practice:

In [ ]:
dirty_path = DATA_DIR / "dirty_marks.csv"
dirty_path.write_text(
    "name,marks\n"
    "  ali , 78\n"
    "SARA,91\n"
    "Ahmed,\n"
    "Omar,-5\n",
    encoding="utf-8",
)

cleaned = []
with open(dirty_path, "r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    for row in reader:
        name = row["name"].strip().title()
        marks_text = row["marks"].strip()
        if marks_text == "" or not marks_text.lstrip("-").isdigit():
            continue                      # skip missing / non-numeric
        marks = int(marks_text)
        if marks < 0 or marks > 100:
            continue                      # skip out-of-range
        cleaned.append({"name": name, "marks": marks})

print("Cleaned from file:", cleaned)

Note the small upgrade: `marks_text.lstrip("-").isdigit()` also accepts
a leading minus sign *so the range check can reject negatives with a
clearer cause* — checking the sign only in `isdigit()` would silently
treat `"-5"` as non-numeric instead of out-of-range.

<a id="6"></a>

## 6. Aggregation and summary statistics

### Concept

Aggregation answers questions about the *whole* collection: How many?
How much overall? What is typical? Best? Worst? Each is an accumulator
loop (Module 6), and `sum()`, `len()`, `min()`, `max()` are the built-in
shortcuts for the common ones.

### Beginner example — total, average, minimum, maximum

In [ ]:
students = [
    {"name": "Ali", "marks": 78},
    {"name": "Sara", "marks": 91},
    {"name": "Ahmed", "marks": 65},
    {"name": "Hina", "marks": 84},
]

marks = []                          # collect one field from every record
for student in students:
    marks.append(student["marks"])

count = len(marks)
total = sum(marks)
average = total / count
print("Count:  ", count)
print("Total:  ", total)
print("Average:", round(average, 1))
print("Minimum:", min(marks))
print("Maximum:", max(marks))

Two-step pattern: (1) gather the field of interest into a plain list,
(2) reduce that list with the built-ins. Forgetting `count == 0` is a
real bug — division by zero — so a report function should guard it (the
project below does).

### Practical example — the full record behind min and max

`min()`/`max()` on the mark list gives the *value*, but reports usually
want the *student*. Track the record instead:

In [ ]:
students = [
    {"name": "Ali", "marks": 78},
    {"name": "Sara", "marks": 91},
    {"name": "Ahmed", "marks": 65},
    {"name": "Hina", "marks": 84},
]

best_student = None
worst_student = None
for student in students:
    if best_student is None or student["marks"] > best_student["marks"]:
        best_student = student      # new best so far
    if worst_student is None or student["marks"] < worst_student["marks"]:
        worst_student = student     # new worst so far

print("Best: ", best_student["name"], "-", best_student["marks"])
print("Worst:", worst_student["name"], "-", worst_student["marks"])

The running-champion pattern from Module 13, applied to records: keep
the current best record; replace it whenever a better one appears. The
`is None` first check handles the very first record.

### Counting categories

In [ ]:
students = [
    {"name": "Ali", "marks": 78},
    {"name": "Sara", "marks": 91},
    {"name": "Ahmed", "marks": 45},
    {"name": "Hina", "marks": 84},
    {"name": "Omar", "marks": 39},
    {"name": "Zoya", "marks": 67},
]

grade_counts = {"A": 0, "B": 0, "C": 0, "F": 0}   # tally per category
for student in students:
    marks = student["marks"]
    if marks >= 80:
        grade_counts["A"] += 1
    elif marks >= 60:
        grade_counts["B"] += 1
    elif marks >= 50:
        grade_counts["C"] += 1
    else:
        grade_counts["F"] += 1

for grade in grade_counts:                        # insertion order: A, B, C, F
    print(grade, "-", grade_counts[grade])

Counting categories is *the* dictionary accumulator: keys are the
categories, values are running tallies (Module 8's frequency-counting
pattern). Starting every tally at 0 means no `KeyError` later.

### Challenge example — a reusable summary function

Wrap the statistics into one function returning a dictionary — a
"summary record" computed *from* records:

In [ ]:
def summarize_marks(records):
    """Return summary statistics for a list of {'name', 'marks'} records."""
    if len(records) == 0:
        return {"count": 0, "average": None, "highest": None, "lowest": None}
    marks = []
    for record in records:
        marks.append(record["marks"])
    return {
        "count": len(marks),
        "average": round(sum(marks) / len(marks), 1),
        "highest": max(marks),
        "lowest": min(marks),
    }


students = [
    {"name": "Ali", "marks": 78},
    {"name": "Sara", "marks": 91},
    {"name": "Ahmed", "marks": 65},
]

summary = summarize_marks(students)
print("Summary:", summary)

empty = summarize_marks([])
print("Empty: ", empty)

Returning `None` (not crashing) for the empty case makes the function
safe to call anywhere — a deliberate design choice you will meet again
as "defensive programming" in Module 16.

<a id="7"></a>

## 7. JSON

### Concept

**JSON — JavaScript Object Notation** — is the standard text format for
sending data between programs (web apps, APIs, config files). It maps
almost perfectly onto Python:

| JSON | Python |
|---|---|
| object `{"key": value}` | dictionary |
| array `[...]` | list |
| string `"..."` | `str` |
| number `78`, `3.5` | `int` / `float` |
| `true` / `false` | `True` / `False` |
| `null` | `None` |

Crucially, JSON **keeps types** — the number `78` round-trips as the
integer `78`, unlike CSV where everything is text.

### Why do we need it?

Datasets, API responses, configuration and NoSQL-style exports are
routinely JSON. `json` is in the standard library — no installs.

### The four functions

| Function | Direction | Works on |
|---|---|---|
| `json.dumps(python_value)` | Python → JSON **s**tring | strings |
| `json.loads(json_text)` | JSON **s**tring → Python | strings |
| `json.dump(python_value, file)` | Python → JSON file | files |
| `json.load(file)` | JSON file → Python | files |

Memory trick: the trailing **s** means **s**tring.

### Beginner example — dumps and loads

In [ ]:
import json

student = {"name": "Ali", "marks": 78, "passed": True, "nickname": None}

as_text = json.dumps(student)         # Python dict -> JSON string
print("JSON text:", as_text)
print("Type:     ", type(as_text))

back_to_python = json.loads(as_text)  # JSON string -> Python dict
print("Back:     ", back_to_python)
print("Type:     ", type(back_to_python))
print("Marks type preserved:", type(back_to_python["marks"]).__name__)

**Explanation:** `dumps` produced a *string* whose contents happen to be
JSON (`true`/`null` lowercase — JSON rules, not Python). `loads` parsed
it back; `78` returned as `int` — the type preservation CSV cannot give
you. Note the pretty differences in spacing: JSON uses `", "` between
pairs; Python's dict `repr` uses `', '`.

### Practical example — dump and load with files

In [ ]:
json_path = DATA_DIR / "students.json"

records = [
    {"name": "Ali", "roll_no": "DS-101", "marks": 78, "subjects": ["math", "cs"]},
    {"name": "Sara", "roll_no": "DS-102", "marks": 91, "subjects": ["math", "cs", "stats"]},
]

# Python -> JSON file
with open(json_path, "w", encoding="utf-8") as file:
    json.dump(records, file, indent=4)      # indent=4 makes it human-readable

# JSON file -> Python
with open(json_path, "r", encoding="utf-8") as file:
    loaded = json.load(file)

print("Loaded", len(loaded), "records; first one:")
print(loaded[0])
print("Sara's subjects:", loaded[1]["subjects"])

`indent=4` writes multi-line, indented JSON — invaluable for debugging;
omit it for compact machine-to-machine files. `json.load` rebuilt the
*entire* nested structure: lists of dictionaries with lists inside.

### Challenge example — CSV vs JSON for the same data

Same records through both formats; watch what happens to types:

In [ ]:
person = {"name": "Ali", "marks": 78, "active": True}

# CSV route: write and read back.
with open(DATA_DIR / "person.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=["name", "marks", "active"])
    writer.writeheader()          # without this line the data row
    writer.writerow(person)       # becomes the header and rows are lost
with open(DATA_DIR / "person.csv", "r", newline="", encoding="utf-8") as file:
    from_csv = next(csv.DictReader(file))       # first row

# JSON route: write and read back.
with open(DATA_DIR / "person.json", "w", encoding="utf-8") as file:
    json.dump(person, file)
with open(DATA_DIR / "person.json", "r", encoding="utf-8") as file:
    from_json = json.load(file)

print("CSV  round-trip:", from_csv, "-> marks is", type(from_csv["marks"]).__name__)
print("JSON round-trip:", from_json, "-> marks is", type(from_json["marks"]).__name__)

The lesson in one line: **CSV is text — every value comes back a
string; JSON keeps types.** Choose CSV for spreadsheet interchange,
JSON when types and structure matter. (Even CSV's `"True"` is a string
here — booleans need explicit conversion from CSV.)

<a id="8"></a>

## 8. Project — Student Performance Data Analyzer

Time to combine everything: a complete data program built in stages,
using only the standard library.

**Problem:** given a CSV of student marks that may contain messy rows,
produce a clean analysis (statistics, grade distribution, top and
struggling students) and save a JSON summary.

**I/O specification:**

- **Input:** `students.csv` with header `name,roll_no,marks` (marks may
  be missing, non-numeric or out of range; names may be dirty).
- **Output:** printed report + `summary.json` with the statistics.

**Algorithm (from Module 13's methodology):**

```text
1. Load records from CSV      -> load_records(path)
2. Validate + clean each row  -> clean_record(row)      (row -> record or None)
3. Compute statistics         -> summarize(records)
4. Count grade categories     -> count_grades(records)
5. Pick top / at-risk records -> top_and_at_risk(records)
6. Print the report           -> print_report(records, summary, grades, ...)
7. Save summary as JSON       -> save_summary(summary, path)
```

One function per stage, `main()` composing them — the decomposition and
function-composition habits from Modules 9 and 13.

### Step 0 — create the sample data file

In [ ]:
analyzer_dir = DATA_DIR / "analyzer"
analyzer_dir.mkdir(exist_ok=True)

csv_file = analyzer_dir / "students.csv"
csv_file.write_text(
    "name,roll_no,marks\n"
    "Ali,DS-101,78\n"
    "  sara ,DS-102,91\n"
    "Ahmed,DS-103,\n"
    "Hina,DS-104,84\n"
    "Omar,DS-105,999\n"
    "Zoya,DS-106,45\n"
    "Bilal,DS-107,67\n"
    "Noor,DS-108,abc\n",
    encoding="utf-8",
)
print("Sample data written to", csv_file.as_posix())

Eight rows: three clean, three dirty in different ways (missing marks,
impossible marks, non-numeric marks), plus a name needing normalising.

### Steps 1–2 — load and clean

In [ ]:
def load_records(path):
    """Read a students CSV file and return (clean_records, rejected_rows).

    A record is accepted only when name is present and marks is an
    integer in 0..100. Rejected rows are returned with the reason.
    """
    clean_records = []
    rejected = []
    with open(path, "r", newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)
        for row in reader:
            record = clean_record(row)
            if record is None:
                rejected.append(row)
            else:
                clean_records.append(record)
    return clean_records, rejected


def clean_record(row):
    """Convert one raw CSV row into a clean record, or None if invalid."""
    name = row["name"].strip().title()
    marks_text = row["marks"].strip()

    if name == "" or marks_text == "":
        return None                          # missing data
    if not marks_text.isdigit():
        return None                          # not a plain number
    marks = int(marks_text)
    if marks < 0 or marks > 100:
        return None                          # impossible marks
    return {"name": name, "roll_no": row["roll_no"].strip(), "marks": marks}


records, rejected = load_records(csv_file)
print("Accepted", len(records), "records; rejected", len(rejected), "rows")
for record in records:
    print("  ", record)

Note the *design*: `clean_record` returns the record **or `None`** —
the "sentinel return" from Module 9 — so `load_records` stays a simple
loop and all validation logic lives in one testable function.

### Steps 3–5 — analyse

In [ ]:
def summarize(records):
    """Return {'count', 'average', 'highest', 'lowest'} for the records."""
    if len(records) == 0:
        return {"count": 0, "average": None, "highest": None, "lowest": None}
    marks = []
    for record in records:
        marks.append(record["marks"])
    return {
        "count": len(marks),
        "average": round(sum(marks) / len(marks), 1),
        "highest": max(marks),
        "lowest": min(marks),
    }


def count_grades(records):
    """Return a grade -> number-of-students tally (A >= 80, B >= 60, C >= 50, F)."""
    counts = {"A": 0, "B": 0, "C": 0, "F": 0}
    for record in records:
        marks = record["marks"]
        if marks >= 80:
            counts["A"] += 1
        elif marks >= 60:
            counts["B"] += 1
        elif marks >= 50:
            counts["C"] += 1
        else:
            counts["F"] += 1
    return counts


def top_and_at_risk(records):
    """Return (names of students with marks >= 80, names with marks < 50)."""
    top = []
    at_risk = []
    for record in records:
        if record["marks"] >= 80:
            top.append(record["name"])
        elif record["marks"] < 50:
            at_risk.append(record["name"])
    return top, at_risk


summary = summarize(records)
grades = count_grades(records)
top, at_risk = top_and_at_risk(records)

print("Summary:", summary)
print("Grades: ", grades)
print("Top:    ", top)
print("At risk:", at_risk)

All three analysers are accumulators you have written before — the only
new thing is that each returns its result so `main` can compose them.

### Step 6 — the report

In [ ]:
def print_report(records, summary, grades, top, at_risk, rejected):
    """Print a human-readable report of the analysis."""
    print("=" * 46)
    print("STUDENT PERFORMANCE REPORT")
    print("=" * 46)
    print(f"{'Name':<8}{'Roll':<10}{'Marks':>5}")
    print("-" * 46)
    for record in records:
        print(f"{record['name']:<8}{record['roll_no']:<10}{record['marks']:>5}")
    print("-" * 46)
    print("Records accepted:", summary["count"], " rejected:", len(rejected))
    if summary["count"] > 0:
        print("Average:", summary["average"], " Highest:", summary["highest"], " Lowest:", summary["lowest"])
    print("Grades:", grades)
    print("Top students:", ", ".join(top) if len(top) > 0 else "(none)")
    print("At risk:     ", ", ".join(at_risk) if len(at_risk) > 0 else "(none)")
    print("=" * 46)


print_report(records, summary, grades, top, at_risk, rejected)

Reporting rejected counts *in* the report keeps the data's provenance
visible — an analysis that hides what it threw away is not trustworthy.

### Step 7 — save the summary as JSON

In [ ]:
summary_file = analyzer_dir / "summary.json"
with open(summary_file, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)

print("Saved summary to", summary_file.as_posix())
print(summary_file.read_text(encoding="utf-8"))

### Step 8 — compose everything in `main()`

The final program wires the stages together. As a *script* it would end
with `main()` under the `if __name__ == "__main__":` guard introduced in
Module 10 — here we call it directly so the notebook runs cleanly.

In [ ]:
def main():
    """Run the full analyze-and-report pipeline."""
    records, rejected = load_records(csv_file)
    summary = summarize(records)
    grades = count_grades(records)
    top, at_risk = top_and_at_risk(records)
    print_report(records, summary, grades, top, at_risk, rejected)
    with open(summary_file, "w", encoding="utf-8") as file:
        json.dump(summary, file, indent=4)
    print("Summary saved to", summary_file.as_posix())


main()

**Sample run** (as a standalone script, the session would look like):

```text
$ python analyzer.py
==============================================
STUDENT PERFORMANCE REPORT
==============================================
Name    Roll      Marks
----------------------------------------------
Ali     DS-101       78
...
Summary saved to summary.json
```

**Possible improvements** (each is within your current abilities):

- Write the *rejected* rows to a `rejected.csv` for audit.
- Add per-subject files and compute subject-wise averages.
- Let the CSV path come from `sys.argv` (Module 10) instead of being fixed.
- Convert `clean_record` to raise `ValueError` with a reason, and have
  `load_records` catch it (Module 12) to record *why* each row failed.
- Sort the report by marks (Module 7's `sort(key=...)`).

<a id="9"></a>

## 9. Lab 15 — CSV/JSON data processing

**Learning objectives:** load real CSV data; clean it with core Python;
summarise, filter and count; persist results as JSON.

**Theory recap (5 minutes):** records = list of dictionaries; cleaning =
strip → check → convert → range-check; `csv.DictReader` gives records
as string-valued dictionaries; `json` keeps types.

### Warm-up (5–10 minutes)

1. Given `r = {"name": "Ali", "marks": 78}`, print `"Ali scored 78"`.
2. Given `rows = [["Ali", "78"], ["Sara", "91"]]`, build the records
   list with integer marks.
3. Print how many records in that list have marks above 80.

### Guided exercises (with hints)

**G1.** Write `load_names(path)` that reads a CSV with header
`name,marks` and returns just the cleaned, title-cased names.

**G2.** Write `marks_above(records, threshold)` returning the names of
students above `threshold`. Test it with 50, 70 and 90 — three filters,
one function (generalisation, Module 16 preview).

**G3.** Write `save_records(records, path)` that writes any records list
to CSV with `DictWriter`, and round-trip your own list through it.

### Independent exercises

**I1.** Load `students.csv` from §8 and compute the *median* mark (sort
the marks; the middle one — for an even count, the average of the two
middle values).

**I2.** Count how many records were rejected *per reason* (missing,
non-numeric, out-of-range). You will need `clean_record` to say why it
rejected a row — refactor it to return `(record, reason)` pairs or to
collect reasons.

**I3.** Build a `roll_no -> name` lookup dictionary from the records,
then answer roll-number queries in constant time (Module 16 will name
what you just discovered).

**I4.** Write the report to a text file `report.txt` instead of printing
it (Module 11 skill), keeping the print for the console.

### Challenge

**C1.** Extend the analyzer: subjects! Load a CSV with columns
`name,math,cs,stats`, clean each subject mark, and report the average
per subject and the best student per subject.

**C2.** Merge two CSVs — `students.csv` and `courses.csv` (with header
`roll_no,course`) — into records of the shape
`{"name": ..., "roll_no": ..., "course": ..., "marks": ...}` and save
the merged data as JSON.

**Expected learning outcomes:** you can take a raw CSV from a
spreadsheet export and turn it into a trustworthy, typed, summarised
dataset using nothing but the standard library.

<a id="10"></a>

## 10. Common mistakes

```python
# 1. Forgetting that CSV values are strings.
with open("students.csv") as file:
    reader = csv.DictReader(file)
    for row in reader:
        if row["marks"] > 80:          # TypeError: '>' not supported
            ...                        # between 'str' and 'int'
# Fix: if int(row["marks"]) > 80:
```

```python
# 2. Forgetting newline="" when writing CSV.
with open("students.csv", "w") as file:     # blank rows on Windows!
    writer = csv.writer(file)
    writer.writerow(["name", "marks"])
# Fix: open("students.csv", "w", newline="", encoding="utf-8")
```

```python
# 3. Modifying the list while looping over it.
for student in students:
    if student["marks"] < 50:
        students.remove(student)       # skips the record after each removal
# Fix: build a new list of keepers, then replace.
#      passing = []
#      for student in students:
#          if student["marks"] >= 50:
#              passing.append(student)
#      students = passing
```

```python
# 4. json.load on a file that was written with json.dumps.
text = json.dumps(records)             # a STRING, not a file
# ... later ...
with open("students.json") as file:
    data = json.load(file)             # fine — but people 'fix' it wrongly:
    data = json.load(text)             # TypeError: str has no .read
# Fix: json.loads(text) for strings, json.load(file) for files.
```

```python
# 5. Dividing by zero on an empty dataset.
average = sum(marks) / len(marks)      # ZeroDivisionError when empty
# Fix: guard first: if len(marks) == 0: return None  (or handle explicitly)
```

```python
# 6. Assuming keys exist in every record.
for record in records:
    print(record["grade"])             # KeyError if a record lacks 'grade'
# Fix: record.get("grade", "n/a")  or validate records on load.
```

<a id="11"></a>

## 11. Exercises

### Quick Review

1. Why is a list of dictionaries a good representation for tabular data?
2. What does `csv.DictReader` give you that `csv.reader` does not?
3. Why must you pass `newline=""` when opening a CSV file for writing?
4. What is the difference between `json.load` and `json.loads`?
5. A CSV row comes back as `{"marks": "78"}`. What is wrong and how do
   you fix it?
6. Name the four cleaning steps in the recommended order.
7. How do you count how many records fall in each grade category?
8. Why should a data program report rejected rows instead of skipping
   them silently?
9. When is JSON preferable to CSV? When is CSV preferable?
10. Why does `clean_record` return `None` for bad rows, and what would
    be an alternative design?

### Predict the Output

```python
# 1.
records = [{"n": "A", "m": 5}, {"n": "B", "m": 8}]
total = 0
for record in records:
    total = total + record["m"]
print(total, total / len(records))
```

```python
# 2.
import json
text = '{"a": 1, "b": [2, 3]}'
data = json.loads(text)
print(data["b"][1] + data["a"])
```

```python
# 3.
names = ["  ali ", "SARA"]
print([name.strip().title() for name in names])   # puzzle: is this taught? it is a for-loop in disguise
# (If unsure, rewrite with an explicit loop and predict that.)
```

```python
# 4.
d = {"A": 0, "B": 0}
for grade in ["A", "B", "A", "A"]:
    d[grade] += 1
print(d)
```

```python
# 5.
import json
print(json.dumps([1, "two", True, None]))
```

<details>
<summary>Predict the Output — answers (click to expand)</summary>

1. `13 6.5`
2. `4`  (b[1] is 3, plus a=1)
3. `['Ali', 'Sara']` — the expression builds a new list of the cleaned
   names. The bracketed form is a *list comprehension*; the explicit
   equivalent is `result = []` then `for name in names: result.append(...)`.
   Full coverage is coming in later modules — recognise the shape now.
4. `{'A': 3, 'B': 1}`
5. `[1, "two", true, null]` — JSON rules: lowercase `true`, `null`.
</details>

### Find the Error

```python
# 1. Intent: read marks and print the total.
import csv
total = 0
with open("students.csv") as file:
    for row in csv.DictReader(file):
        total += row["marks"]
print(total)
```

```python
# 2. Intent: write one record to CSV.
import csv
with open("out.csv", "w") as file:
    writer = csv.writer(file)
    write_row(["name", "marks"])
    writer.writerow(["Ali", 78])
```

```python
# 3. Intent: load JSON from disk.
import json
with open("summary.json") as file:
    summary = json.loads(file)
print(summary["average"])
```

```python
# 4. Intent: remove failing students.
students = [{"n": "Ali", "m": 78}, {"n": "Omar", "m": 42},
            {"n": "Sara", "m": 91}, {"n": "Zoya", "m": 30}]
for student in students:
    if student["m"] < 50:
        students.remove(student)
print(students)
```

```python
# 5. Intent: average marks.
marks = []
for record in records:          # records exists and is non-empty
    marks.append(record["marks"])
average = sum(marks) / len(marks)
# ...but the same code ran earlier on an EMPTY records list and crashed.
# Why, and what is the robust pattern?
```

<details>
<summary>Find the Error — answers (click to expand)</summary>

1. `row["marks"]` is the string `"78"`, so `total += "78"` concatenates
   (or raises `TypeError` on the first row). Fix: `total += int(row["marks"])`.
2. `write_row` is not a name — the method is `writer.writerow(...)`.
   Also prefer `newline=""` and `encoding="utf-8"` in the `open`.
3. `json.loads` takes a *string*; for a file object use `json.load(file)`
   (or `json.loads(file.read())`).
4. Removing while looping skips every second failing record (the loop
   index jumps over the element shifted into place). Fix: build a keep
   list, or loop over a copy (`for student in students[:]`) — but the
   keep-list is the honest beginner fix.
5. Empty input → `len(marks)` is 0 → `ZeroDivisionError`. Robust pattern:
   guard before dividing (`if len(marks) == 0: ... else: ...`) — exactly
   what `summarize()` does in §6.
</details>

### Programming Exercises

1. **Record builder** — given `raw = "Ali,DS-101,78"`, build the record
   dictionary `{"name": "Ali", "roll_no": "DS-101", "marks": 78}`.
2. **Field extractor** — write `all_names(records)` returning a list of
   names in record order.
3. **Filter by grade** — write `students_with_grade(records, "B")`
   returning records whose marks fall in the B band (60–79).
4. **Class statistics** — for a records list print count, average, and
   the marks that are above average (two passes: compute, then filter).
5. **CSV loader** — write `load_students(path)` returning clean records
   with integer marks; test it on `file_sandbox/students.csv`.
6. **CSV saver** — write `save_students(records, path)` with
   `DictWriter`; load your file back and compare with the original list.
7. **Category counter** — write `count_by(records, key)` counting how
   many records share each value of `key`; test on a records list that
   has a `"course"` field.
8. **Grade distribution report** — print a bar of `*` per student in
   each grade (e.g. `A: **` for two A students).
9. **JSON resume** — save your records to JSON with `indent=4`, load it
   back, and print only students with marks ≥ 80.
10. **Full pipeline** — write `analyze(path)` that loads, cleans,
    summarises and *returns* the summary dictionary; call it on a CSV
    you create with at least two dirty rows.

### Challenge Problems

1. **Subject report** — records of the shape
   `{"name": ..., "marks": {"math": ..., "cs": ...}}`: print each
   subject's average and each student's best subject.
2. **Dirty-data duel** — construct a CSV with one row broken in *each*
   way from §5's table; make your analyzer classify every rejection and
   write a `rejections.csv` audit file.
3. **JSON → CSV converter** — read a JSON file of records and write a
   CSV of the same data (explicitly converting every non-string value);
   round-trip it back and verify types match your expectations.

<a id="12"></a>

## 12. Summary

- **Structured data** = same fields for every item → represent as a
  **list of dictionaries** (table = list, row = dict, column = key).
- The five processing patterns — **filter, search, transform, aggregate,
  count** — are accumulator loops you already own.
- **CSV** via `csv.reader` (rows as lists) and `csv.DictReader` (rows as
  dictionaries); write with `csv.writer`/`DictWriter`; always
  `newline=""`, and remember **CSV values are strings**.
- **Cleaning** = strip → check for missing → check numeric → convert →
  range-check → de-duplicate, with rejections *reported*, not hidden.
- **JSON** keeps types: `dumps`/`loads` for strings, `dump`/`load` for
  files; `indent=4` for readable files.
- The **analyzer project** decomposed a data job into load → clean →
  analyse → report → save, each stage one function — the architecture
  of every data pipeline you will ever write.

**Next:** [Module 16 — Computational Thinking](16_Computational_Thinking.ipynb)
steps back from syntax to *how to think*: abstraction, decomposition,
choosing data structures, and what O(1), O(n) and O(n²) really mean.